# Version 2 Transformer Data Preparation and Tokenization

This notebook prepares the leakage-safe 2024 data design for the DistilBERT challenger. It reproduces the locked Version 1 remediation, development/final-test split, five development folds, and canonical eight-class mapping before auditing token lengths on development data only.

Guardrails: use only the local cleaned 2024 source; do not initialize or train a classification model; do not evaluate the final internal test; do not access 2025 or 2026 data; and do not display or export complaint narratives, row-level hashes, token IDs, or row-level token lengths.

## 1. Environment, paths, and locked constants

In [1]:
from pathlib import Path
import hashlib
import platform
import sys

import numpy as np
import pandas as pd
import sklearn
import tokenizers
import torch
import transformers
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoTokenizer, DataCollatorWithPadding


def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        expected = candidate / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"
        if expected.exists() and (candidate / ".gitignore").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository and locked 2024 source.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"

EXPECTED_ENVIRONMENT = {
    "Python": "3.11.15",
    "Pandas": "3.0.3",
    "NumPy": "2.4.6",
    "Scikit-learn": "1.9.0",
    "Transformers": "4.57.6",
    "Tokenizers": "0.22.2",
    "PyTorch base": "2.9.1",
}
torch_runtime_version = torch.__version__
conda_environment = Path(sys.prefix).name
observed_environment = {
    "Python": platform.python_version(),
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Scikit-learn": sklearn.__version__,
    "Transformers": transformers.__version__,
    "Tokenizers": tokenizers.__version__,
    "PyTorch base": torch_runtime_version.split("+")[0],
}

EXPECTED_SOURCE_SIZE = 54_908_639
EXPECTED_SOURCE_SHA256 = "b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa7d68f09d657cdfd919"
EXPECTED_SOURCE_ROWS = 50_000
EXPECTED_CONFLICTING_GROUPS = 74
EXPECTED_LOCKED_CONFLICT_ROWS = 1_780
EXPECTED_SAME_LABEL_ROWS_REMOVED = 14_374
EXPECTED_MODELING_ROWS = 33_042
EXPECTED_DEVELOPMENT_ROWS = 26_433
EXPECTED_FINAL_TEST_ROWS = 6_609
OUTER_SPLITS = 5
DEVELOPMENT_SPLITS = 5
FINAL_TEST_FOLD = 0
RANDOM_STATE = 42

CANONICAL_LABELS = (
    "Checking or savings account",
    "Credit card",
    "Credit reporting or other personal consumer reports",
    "Debt collection",
    "Money transfer, virtual currency, or money service",
    "Mortgage",
    "Student loan",
    "Vehicle loan or lease",
)
label2id = {label: label_id for label_id, label in enumerate(CANONICAL_LABELS)}
id2label = {label_id: label for label, label_id in label2id.items()}

TOKENIZER_ID = "distilbert/distilbert-base-uncased"
TOKENIZER_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"
TOKEN_LENGTH_CANDIDATES = (128, 256)
MINIMUM_COVERAGE_PERCENT = 95.0

if conda_environment != "complaint-v2":
    raise RuntimeError(f"Wrong Conda environment: {conda_environment}")
if observed_environment != EXPECTED_ENVIRONMENT:
    raise RuntimeError(f"Version 2 environment mismatch: {observed_environment}")

print("Project root located: PASS")
print(f"Input dataset: {DATA_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Conda environment: {conda_environment}")
for package_name, version in observed_environment.items():
    print(f"{package_name}: {version}")
print(f"PyTorch runtime: {torch_runtime_version}")
print(f"Locked categories: {len(CANONICAL_LABELS)}")
print("Environment and locked constants: PASS")

Project root located: PASS
Input dataset: data/processed/cfpb_complaints_2024_cleaned.csv
Conda environment: complaint-v2
Python: 3.11.15
Pandas: 3.0.3
NumPy: 2.4.6
Scikit-learn: 1.9.0
Transformers: 4.57.6
Tokenizers: 0.22.2
PyTorch base: 2.9.1
PyTorch runtime: 2.9.1+cu126
Locked categories: 8
Environment and locked constants: PASS


## 2. Source integrity and required columns

The file is fingerprinted before it is parsed. Only the two required modeling columns are loaded, and their values are never displayed.

In [2]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Locked source not found: {DATA_PATH}")

source_size = DATA_PATH.stat().st_size
source_digest = hashlib.sha256()
with DATA_PATH.open("rb") as source_handle:
    for chunk in iter(lambda: source_handle.read(1024 * 1024), b""):
        source_digest.update(chunk)
source_sha256 = source_digest.hexdigest()

if source_size != EXPECTED_SOURCE_SIZE:
    raise RuntimeError(f"Source-size mismatch: {source_size}")
if source_sha256 != EXPECTED_SOURCE_SHA256:
    raise RuntimeError("Source SHA-256 does not match the locked fingerprint.")

required_columns = ["clean_complaint_text", "product"]
source_df = pd.read_csv(DATA_PATH, usecols=required_columns)
if set(source_df.columns) != set(required_columns):
    raise RuntimeError(f"Required-column mismatch: {list(source_df.columns)}")
if len(source_df) != EXPECTED_SOURCE_ROWS:
    raise RuntimeError(f"Source-row mismatch: {len(source_df)}")
if source_df[required_columns].isna().any().any():
    raise RuntimeError("Required modeling columns contain missing values.")

blank_text_rows = int(source_df["clean_complaint_text"].astype(str).str.strip().eq("").sum())
blank_label_rows = int(source_df["product"].astype(str).str.strip().eq("").sum())
if blank_text_rows or blank_label_rows:
    raise RuntimeError("Required modeling columns contain blank values.")

accessed_data_paths = {DATA_PATH.resolve()}

print("Locked source integrity:")
print(f"- Path: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"- File size: {source_size:,} bytes (PASS)")
print("- SHA-256: matches the committed fingerprint (PASS)")
print(f"- Source rows: {len(source_df):,}")
print(f"- Required columns present: {required_columns}")
print(f"- Missing or blank required values: {blank_text_rows + blank_label_rows:,}")

Locked source integrity:
- Path: data\processed\cfpb_complaints_2024_cleaned.csv
- File size: 54,908,639 bytes (PASS)
- SHA-256: matches the committed fingerprint (PASS)
- Source rows: 50,000
- Required columns present: ['clean_complaint_text', 'product']
- Missing or blank required values: 0


## 3. Locked duplicate-conflict remediation

Conflicting-label groups are identified across the complete cleaned source before the eight-category filter is applied. The first original row is retained for each remaining repeated same-label group.

In [3]:
def normalize_grouping_text(value: object) -> str:
    normalized = " ".join(str(value).strip().split())
    if not normalized:
        raise ValueError("Normalized grouping text must not be empty.")
    return normalized


def stable_text_hash(value: object) -> str:
    return hashlib.sha256(normalize_grouping_text(value).encode("utf-8")).hexdigest()


working_df = source_df[required_columns].copy()
working_df["source_row_order"] = np.arange(len(working_df), dtype=np.int64)
working_df["normalized_text_hash"] = working_df["clean_complaint_text"].map(stable_text_hash)

group_label_counts = working_df.groupby("normalized_text_hash", sort=False)["product"].nunique()
conflicting_hashes = set(group_label_counts[group_label_counts > 1].index)
conflicting_label_groups = len(conflicting_hashes)

locked_scope_df = working_df[working_df["product"].isin(CANONICAL_LABELS)].copy()
locked_conflict_mask = locked_scope_df["normalized_text_hash"].isin(conflicting_hashes)
locked_scope_conflicting_rows = int(locked_conflict_mask.sum())
scope_without_conflicts = locked_scope_df.loc[~locked_conflict_mask].copy()

remediated_df = scope_without_conflicts.drop_duplicates(
    subset=["normalized_text_hash", "product"],
    keep="first",
).reset_index(drop=True)
same_label_rows_removed = len(scope_without_conflicts) - len(remediated_df)
original_order_preserved = bool(remediated_df["source_row_order"].is_monotonic_increasing)

assert conflicting_label_groups == EXPECTED_CONFLICTING_GROUPS
assert locked_scope_conflicting_rows == EXPECTED_LOCKED_CONFLICT_ROWS
assert same_label_rows_removed == EXPECTED_SAME_LABEL_ROWS_REMOVED
assert len(remediated_df) == EXPECTED_MODELING_ROWS
assert remediated_df["normalized_text_hash"].is_unique
assert int(remediated_df.groupby("normalized_text_hash")["product"].nunique().max()) == 1
assert set(remediated_df["product"].unique()) == set(CANONICAL_LABELS)
assert original_order_preserved

remediation_summary = pd.DataFrame(
    [
        ("Source rows", len(working_df)),
        ("Conflicting-label groups in complete source", conflicting_label_groups),
        ("Locked-scope rows before remediation", len(locked_scope_df)),
        ("Locked-scope conflicting rows excluded", locked_scope_conflicting_rows),
        ("Repeated same-label rows removed", same_label_rows_removed),
        ("Corrected modeling rows", len(remediated_df)),
    ],
    columns=["Measure", "Count"],
)
print(remediation_summary.to_string(index=False))
print(f"Original row order preserved: {original_order_preserved}")
print("No narratives or normalized-text hashes are displayed or saved.")

                                    Measure  Count
                                Source rows  50000
Conflicting-label groups in complete source     74
       Locked-scope rows before remediation  49196
     Locked-scope conflicting rows excluded   1780
           Repeated same-label rows removed  14374
                    Corrected modeling rows  33042
Original row order preserved: True
No narratives or normalized-text hashes are displayed or saved.


## 4. Locked split, development folds, and label mapping

Fold 0 of the outer splitter remains the shared 2024 final internal test. The final-test rows are reconstructed only to validate the locked boundary; they are not token-length audited, tokenized for modeling, scored, or evaluated.

In [4]:
outer_splitter = StratifiedGroupKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
outer_folds = list(
    outer_splitter.split(
        remediated_df["clean_complaint_text"],
        remediated_df["product"],
        groups=remediated_df["normalized_text_hash"],
    )
)
development_indices, final_test_indices = outer_folds[FINAL_TEST_FOLD]
development_df = remediated_df.iloc[development_indices].reset_index(drop=True)
final_test_df = remediated_df.iloc[final_test_indices].reset_index(drop=True)

development_groups = set(development_df["normalized_text_hash"])
final_test_groups = set(final_test_df["normalized_text_hash"])
development_final_overlap = len(development_groups.intersection(final_test_groups))

assert len(development_df) == EXPECTED_DEVELOPMENT_ROWS
assert len(final_test_df) == EXPECTED_FINAL_TEST_ROWS
assert development_final_overlap == 0
assert set(development_df["product"].unique()) == set(CANONICAL_LABELS)
assert set(final_test_df["product"].unique()) == set(CANONICAL_LABELS)

development_splitter = StratifiedGroupKFold(
    n_splits=DEVELOPMENT_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
development_folds = list(
    development_splitter.split(
        development_df["clean_complaint_text"],
        development_df["product"],
        groups=development_df["normalized_text_hash"],
    )
)

fold_rows = []
validation_assignments = np.zeros(len(development_df), dtype=np.int8)
for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(development_folds):
    fold_train = development_df.iloc[fold_train_indices]
    fold_validation = development_df.iloc[fold_validation_indices]
    fold_overlap = len(
        set(fold_train["normalized_text_hash"]).intersection(
            set(fold_validation["normalized_text_hash"])
        )
    )
    validation_assignments[fold_validation_indices] += 1
    train_class_count = int(fold_train["product"].nunique())
    validation_class_count = int(fold_validation["product"].nunique())
    assert fold_overlap == 0
    assert train_class_count == len(CANONICAL_LABELS)
    assert validation_class_count == len(CANONICAL_LABELS)
    fold_rows.append(
        {
            "Fold": fold_number,
            "Training rows": len(fold_train),
            "Validation rows": len(fold_validation),
            "Group overlap": fold_overlap,
            "Training classes": train_class_count,
            "Validation classes": validation_class_count,
        }
    )

assert len(development_folds) == DEVELOPMENT_SPLITS
assert np.all(validation_assignments == 1)
assert label2id == {label: index for index, label in enumerate(CANONICAL_LABELS)}
assert id2label == {index: label for index, label in enumerate(CANONICAL_LABELS)}

print("Locked outer split:")
print(f"- Development rows: {len(development_df):,}")
print(f"- Final internal-test rows: {len(final_test_df):,}")
print(f"- Development/final-test normalized-text overlap: {development_final_overlap}")
print(f"- Eight categories present in both partitions: PASS")
print("Development-fold validation:")
print(pd.DataFrame(fold_rows).to_string(index=False))
print("Every development row assigned to exactly one validation fold: PASS")
print("Canonical label mapping:")
for label_id in range(len(CANONICAL_LABELS)):
    print(f"  {label_id}: {id2label[label_id]}")

Locked outer split:
- Development rows: 26,433
- Final internal-test rows: 6,609
- Development/final-test normalized-text overlap: 0
- Eight categories present in both partitions: PASS
Development-fold validation:
 Fold  Training rows  Validation rows  Group overlap  Training classes  Validation classes
    0          21146             5287              0                 8                   8
    1          21146             5287              0                 8                   8
    2          21146             5287              0                 8                   8
    3          21147             5286              0                 8                   8
    4          21147             5286              0                 8                   8
Every development row assigned to exactly one validation fold: PASS
Canonical label mapping:
  0: Checking or savings account
  1: Credit card
  2: Credit reporting or other personal consumer reports
  3: Debt collection
  4: Money transfer

## 5. Development-only tokenizer and token-length audit

Only the 26,433 development rows determine the maximum token length. Tokenization includes special tokens with no padding and no truncation. Row-level lengths and token IDs remain in memory and are not displayed or saved.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    revision=TOKENIZER_REVISION,
    use_fast=True,
)
if tokenizer.__class__.__name__ != "DistilBertTokenizerFast":
    raise RuntimeError(f"Unexpected tokenizer class: {tokenizer.__class__.__name__}")
if tokenizer.vocab_size != 30_522 or tokenizer.model_max_length != 512:
    raise RuntimeError("Tokenizer metadata does not match the locked DistilBERT tokenizer.")

token_length_chunks = []
tokenization_batch_size = 256
for start in range(0, len(development_df), tokenization_batch_size):
    stop = min(start + tokenization_batch_size, len(development_df))
    encoded = tokenizer(
        development_df["clean_complaint_text"].iloc[start:stop].tolist(),
        add_special_tokens=True,
        padding=False,
        truncation=False,
        return_length=True,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )
    token_length_chunks.append(np.asarray(encoded["length"], dtype=np.int32))
    del encoded

development_token_lengths = np.concatenate(token_length_chunks)
assert len(development_token_lengths) == EXPECTED_DEVELOPMENT_ROWS
assert int(development_token_lengths.min()) >= 2

def percentage(mask: np.ndarray) -> float:
    return float(np.mean(mask) * 100.0)


token_length_statistics = {
    "Minimum": float(np.min(development_token_lengths)),
    "Mean": float(np.mean(development_token_lengths)),
    "Population standard deviation": float(np.std(development_token_lengths, ddof=0)),
    "Median": float(np.percentile(development_token_lengths, 50)),
    "75th percentile": float(np.percentile(development_token_lengths, 75)),
    "90th percentile": float(np.percentile(development_token_lengths, 90)),
    "95th percentile": float(np.percentile(development_token_lengths, 95)),
    "99th percentile": float(np.percentile(development_token_lengths, 99)),
    "Maximum": float(np.max(development_token_lengths)),
    "At or below 128 (%)": percentage(development_token_lengths <= 128),
    "Above 128 (%)": percentage(development_token_lengths > 128),
    "At or below 256 (%)": percentage(development_token_lengths <= 256),
    "Above 256 (%)": percentage(development_token_lengths > 256),
    "Above 512 (%)": percentage(development_token_lengths > 512),
}

coverage_128 = token_length_statistics["At or below 128 (%)"]
selected_max_length = 128 if coverage_128 >= MINIMUM_COVERAGE_PERCENT else 256
selected_coverage = percentage(development_token_lengths <= selected_max_length)
residual_truncation_percentage = percentage(development_token_lengths > selected_max_length)

assert selected_max_length in TOKEN_LENGTH_CANDIDATES
assert selected_max_length == (128 if coverage_128 >= 95.0 else 256)

tokenizer_metadata = {
    "Tokenizer ID": TOKENIZER_ID,
    "Revision": TOKENIZER_REVISION,
    "Tokenizer class": tokenizer.__class__.__name__,
    "Vocabulary size": tokenizer.vocab_size,
    "Maximum supported length": tokenizer.model_max_length,
    "CLS token ID": tokenizer.cls_token_id,
    "SEP token ID": tokenizer.sep_token_id,
    "PAD token ID": tokenizer.pad_token_id,
    "UNK token ID": tokenizer.unk_token_id,
}

print("Locked tokenizer metadata:")
for name, value in tokenizer_metadata.items():
    print(f"- {name}: {value}")
print("Development token-length statistics:")
for name, value in token_length_statistics.items():
    if name.endswith("(%)"):
        print(f"- {name}: {value:.4f}")
    else:
        print(f"- {name}: {value:.4f}")
print(f"Selected maximum token length: {selected_max_length}")
print(f"Selected-length coverage: {selected_coverage:.4f}%")
print(f"Residual truncation: {residual_truncation_percentage:.4f}%")
print("Selection rule followed: PASS")

Locked tokenizer metadata:
- Tokenizer ID: distilbert/distilbert-base-uncased
- Revision: 12040accade4e8a0f71eabdb258fecc2e7e948be
- Tokenizer class: DistilBertTokenizerFast
- Vocabulary size: 30522
- Maximum supported length: 512
- CLS token ID: 101
- SEP token ID: 102
- PAD token ID: 0
- UNK token ID: 100
Development token-length statistics:
- Minimum: 4.0000
- Mean: 289.0243
- Population standard deviation: 386.8186
- Median: 187.0000
- 75th percentile: 352.0000
- 90th percentile: 601.0000
- 95th percentile: 837.4000
- 99th percentile: 1784.0000
- Maximum: 8136.0000
- At or below 128 (%): 36.9765
- Above 128 (%): 63.0235
- At or below 256 (%): 61.9945
- Above 256 (%): 38.0055
- Above 512 (%): 13.4150
Selected maximum token length: 256
Selected-length coverage: 61.9945%
Residual truncation: 38.0055%
Selection rule followed: PASS


## 6. Locked truncation and dynamic-padding smoke test

The selected maximum length is now locked. A small development-only batch validates truncation and longest-sequence dynamic padding. Only aggregate tensor shapes and lengths are reported.

In [6]:
sample_indices = [0, len(development_df) // 3, (2 * len(development_df)) // 3, len(development_df) - 1]
sample_texts = development_df["clean_complaint_text"].iloc[sample_indices].tolist()
sample_encodings = tokenizer(
    sample_texts,
    add_special_tokens=True,
    truncation=True,
    max_length=selected_max_length,
    padding=False,
    return_attention_mask=True,
    return_token_type_ids=False,
)
sample_features = [
    {
        "input_ids": sample_encodings["input_ids"][row_index],
        "attention_mask": sample_encodings["attention_mask"][row_index],
    }
    for row_index in range(len(sample_indices))
]
pre_padding_lengths = [len(feature["input_ids"]) for feature in sample_features]

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",
    return_tensors="pt",
)
padded_batch = data_collator(sample_features)
input_shape = tuple(int(dimension) for dimension in padded_batch["input_ids"].shape)
attention_shape = tuple(int(dimension) for dimension in padded_batch["attention_mask"].shape)
padded_length = input_shape[1]

assert input_shape == attention_shape
assert input_shape[0] == len(sample_indices)
assert padded_length == max(pre_padding_lengths)
assert padded_length <= selected_max_length
assert padded_batch["attention_mask"].sum(dim=1).tolist() == pre_padding_lengths

print("Development-only dynamic-padding validation:")
print(f"- Batch size: {input_shape[0]}")
print(f"- Pre-padding length range: {min(pre_padding_lengths)} to {max(pre_padding_lengths)} tokens")
print(f"- Padded input tensor shape: {input_shape}")
print(f"- Attention-mask tensor shape: {attention_shape}")
print(f"- Batch padded only to its longest truncated sequence: {padded_length}")
print(f"- Locked maximum token length: {selected_max_length}")
print("Dynamic padding and truncation: PASS")

del sample_texts, sample_encodings, sample_features, padded_batch

Development-only dynamic-padding validation:
- Batch size: 4
- Pre-padding length range: 90 to 256 tokens
- Padded input tensor shape: (4, 256)
- Attention-mask tensor shape: (4, 256)
- Batch padded only to its longest truncated sequence: 256
- Locked maximum token length: 256
Dynamic padding and truncation: PASS


## 7. Final Issue #39 validation

The final table confirms the automated pre-training data and tokenization checks.

**Scope statement:** Static review of the notebook source found no classification-model initialization, training, prediction, final-test token-length audit, or row-level export code. No 2025 or 2026 data-loading code is present.

In [7]:
validation_checks = {
    "Expected Conda environment": conda_environment == "complaint-v2",
    "Pinned package versions": observed_environment == EXPECTED_ENVIRONMENT,
    "Locked source path only": accessed_data_paths == {DATA_PATH.resolve()},
    "Source file size": source_size == EXPECTED_SOURCE_SIZE,
    "Source SHA-256": source_sha256 == EXPECTED_SOURCE_SHA256,
    "Required columns": set(source_df.columns) == set(required_columns),
    "Source row count": len(source_df) == EXPECTED_SOURCE_ROWS,
    "Conflicting-label groups": conflicting_label_groups == EXPECTED_CONFLICTING_GROUPS,
    "Locked-scope conflict rows excluded": locked_scope_conflicting_rows == EXPECTED_LOCKED_CONFLICT_ROWS,
    "Repeated same-label rows removed": same_label_rows_removed == EXPECTED_SAME_LABEL_ROWS_REMOVED,
    "Corrected modeling rows": len(remediated_df) == EXPECTED_MODELING_ROWS,
    "Original row order preserved": original_order_preserved,
    "Development rows": len(development_df) == EXPECTED_DEVELOPMENT_ROWS,
    "Final internal-test rows": len(final_test_df) == EXPECTED_FINAL_TEST_ROWS,
    "Development/final-test overlap is zero": development_final_overlap == 0,
    "Eight categories in both outer partitions": set(development_df["product"]) == set(CANONICAL_LABELS) and set(final_test_df["product"]) == set(CANONICAL_LABELS),
    "Five development folds": len(development_folds) == DEVELOPMENT_SPLITS,
    "Zero group overlap in every development fold": all(row["Group overlap"] == 0 for row in fold_rows),
    "Every development row validates once": bool(np.all(validation_assignments == 1)),
    "Canonical label mapping": label2id == {label: index for index, label in enumerate(CANONICAL_LABELS)},
    "Pinned tokenizer metadata": tokenizer.__class__.__name__ == "DistilBertTokenizerFast" and tokenizer.vocab_size == 30_522 and tokenizer.model_max_length == 512,
    "Development-only length audit complete": len(development_token_lengths) == EXPECTED_DEVELOPMENT_ROWS,
    "95 percent selection rule followed": selected_max_length == (128 if coverage_128 >= 95.0 else 256),
    "Dynamic padding validated": padded_length == max(pre_padding_lengths) and padded_length <= selected_max_length,
}

validation_table = pd.DataFrame(
    [(name, "PASS" if passed else "FAIL") for name, passed in validation_checks.items()],
    columns=["Check", "Result"],
)
failed_checks = [name for name, passed in validation_checks.items() if not passed]
print(validation_table.to_string(index=False))
print(f"Final validation: {len(validation_checks) - len(failed_checks)}/{len(validation_checks)} checks passed")
if failed_checks:
    raise RuntimeError(f"Issue #39 validation failed: {failed_checks}")
print("Issue #39 data-preparation and tokenization gate: PASS")
print("No model initialization, training, final-test evaluation, 2025/2026 access, or row-level export occurred.")

                                       Check Result
                  Expected Conda environment   PASS
                     Pinned package versions   PASS
                     Locked source path only   PASS
                            Source file size   PASS
                              Source SHA-256   PASS
                            Required columns   PASS
                            Source row count   PASS
                    Conflicting-label groups   PASS
         Locked-scope conflict rows excluded   PASS
            Repeated same-label rows removed   PASS
                     Corrected modeling rows   PASS
                Original row order preserved   PASS
                            Development rows   PASS
                    Final internal-test rows   PASS
      Development/final-test overlap is zero   PASS
   Eight categories in both outer partitions   PASS
                      Five development folds   PASS
Zero group overlap in every development fold   PASS
        Ever